In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import Optional
from pydantic import BaseModel, Field
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from difflib import SequenceMatcher

class PIOExtraction(BaseModel):
    population: Optional[str] = Field(description="The patients or problem. Return null if none.")
    intervention: Optional[str] = Field(description="The main treatment. Return null if none.")
    outcome: Optional[str] = Field(description="The primary results. Return null if none.")

llm = OllamaLLM(model="llama3.1", format="json", temperature=0)
parser = JsonOutputParser(pydantic_object=PIOExtraction)

def evaluate_extraction(original_text, extracted_text, gt_mask, threshold=0.6):
    if not extracted_text or str(extracted_text).lower() in ('null', 'none'):
        return "None", None 
        
    items = [item.strip() for item in str(extracted_text).split(';')]
    
    orig_words = original_text.split()
    item_scores = []
    
    for item in items:
        if not item: continue
        
        ext_words = item.split()
        window = len(ext_words)
        best_ratio, best_start = 0, 0
        
        for i in range(len(orig_words) - window + 1):
            window_text = " ".join(orig_words[i:i+window])
            ratio = SequenceMatcher(None, item.lower(), window_text.lower()).ratio()
            if ratio > best_ratio:
                best_ratio, best_start = ratio, i

        if best_ratio < threshold:
            item_scores.append(0.0) # (Hallucination)
        else:
            mask_slice = gt_mask[best_start : best_start + window]
            if any(label == 1 for label in mask_slice):
                item_scores.append(1.0) # (Hit)
            else:
                item_scores.append(0.0) # (Miss)

    if not item_scores:
        return "None", None

    avg_precision = sum(item_scores) / len(item_scores)
    
    if avg_precision == 1.0:
        return "Hit (Relaxed)", 1.0
    elif avg_precision == 0.0:
        return "Hallucinated", 0.0
    else:
        return "Partial Hit", avg_precision

def run_zero_shot_experiment(test_texts, test_masks):
    print(f"\n" + "="*60)
    print(f"RUNNING ZERO-SHOT EXPERIMENT")
    print("="*60)

    template = """You are a medical researcher. Extract PIO elements in JSON format.
    CRITICAL RULE: Your extractions MUST be exact, continuous substrings copied directly from the abstract. Do not add or change any words.
    
    {format_instructions}
    
    --- ACTUAL TASK ---
    Abstract: {abstract}
    Output:"""

    pipeline = PromptTemplate(
        template=template,
        input_variables=["abstract"], 
        partial_variables={
            "format_instructions": parser.get_format_instructions()
        }
    ) | llm | parser

    limit = 1000
    results = []
    
    for i in tqdm(range(limit), desc="Processing Abstracts (Zero-shot)"):
        text = test_texts[i]
        
        try:
            res = pipeline.invoke({
                "abstract": text
            })
            
            row = {"Text": text}
            
            for short_key, full_key in [('Pop', 'population'), ('Int', 'intervention'), ('Out', 'outcome')]:
                extracted = res.get(full_key)
                status, prec = evaluate_extraction(text, extracted, test_masks[short_key][i])
                
                row.update({
                    f"{short_key}_Extracted": extracted,
                    f"{short_key}_Status": status,
                    f"{short_key}_Precision": prec
                })
                
            results.append(row)
                
        except Exception as e:
            print(f"\nError processing sample {i}: {e}") 

    if results:
        df = pd.DataFrame(results)
        filename = "pio_zero_shot_results.csv"
        df.to_csv(filename, index=False)

print("Loading dataset...")
data = np.load('./data/ebm_nlp_2_00/processed/ebm_abstracts_full.npz', allow_pickle=True)
    
test_texts = data['test_texts']
test_masks = {'Pop': data['test_p'], 'Int': data['test_i'], 'Out': data['test_o']}

run_zero_shot_experiment(test_texts, test_masks)


In [4]:
import pandas as pd

def calculate_metrics(file_paths):
    all_summaries = []
    
    for path in file_paths:
        try:
            df = pd.read_csv(path)
            
            name_parts = path.split('_')
            shots = f"{name_parts[2]}-Shot" if len(name_parts) > 2 else path
            
            summary = {"Experiment": shots}
            
            for pio in ['Pop', 'Int', 'Out']:
                strict_accuracy = df[f'{pio}_Precision'].mean()
                
                stats = df[f'{pio}_Status'].value_counts()
                hits = stats.get('Hit (Relaxed)', 0)
                partial = stats.get('Partial Hit', 0)
                misses = stats.get('Miss', 0)
                halluc = stats.get('Hallucinated', 0)
                
                summary[f"{pio}_Acc"] = f"{strict_accuracy*100:.1f}%"
                summary[f"{pio}_Counts (H/P/M/Halluc)"] = f"{hits}/{partial}/{misses}/{halluc}"
                
            all_summaries.append(summary)
        except Exception as e:
            print(f"Error processing {path}: {e}")
            
    return pd.DataFrame(all_summaries)

my_files = [
    "pio_zero_shot_results.csv", 
]

report_table = calculate_metrics(my_files)
print(report_table.to_string(index=False))

Experiment Pop_Acc Pop_Counts (H/P/M/Halluc) Int_Acc Int_Counts (H/P/M/Halluc) Out_Acc Out_Counts (H/P/M/Halluc)
 shot-Shot   85.0%                 85/0/0/15   91.9%                  91/0/0/8   73.0%                 73/0/0/27
